# M2 · Segmentación de instancias de vehículos (YOLO11s-seg) con dataset público
**IAM-811 · Laboratorio Final — Sistema Inteligente de Control de Peaje**

| | |
|---|---|
| **Dataset** | [*Vehicle Segmentation*](https://universe.roboflow.com/akash-bauri-nyofc/vehicle-segmentation) — Akash Bauri, Roboflow Universe |
| **Licencia** | CC BY 4.0 (se cita en `entrenamiento/DATASETS.md` y en el README) |
| **Versión / formato** | v4 · `yolov11` (polígonos de segmentación) |
| **Categorías del peaje** | `auto` · `minibus` · `bus` · `camion` (mínimo 3 con ≥ 30 instancias en train) |
| **Salida** | `m2_vehiculos_seg.pt` → `MyDrive/peaje-ia/pesos/` → `inferencia/pesos/` del Pod de RunPod |
| **Métricas** | mAP50 y mAP50-95 de caja y máscara (global y por clase), matriz de confusión y **exactitud en las 22 fotos bolivianas** del repositorio |

> **Ojo con la clase `auto` del dataset:** el dataset es de tráfico de la India y allí `auto` es el
> **auto-rickshaw (tuk-tuk)**, no un automóvil. Se descarta; los automóviles vienen de `car`, `sedan`, `suv`, etc.
> La sección 3 lo muestra visualmente antes de remapear.

Orden de ejecución: *Entorno de ejecución → Ejecutar todas* (GPU T4). Pide la API key solo si no existe el
secreto `ROBOFLOW_API_KEY`.

## 1. Instalación

In [ ]:
!nvidia-smi -L || echo "ACTIVE LA GPU T4: Entorno de ejecución > Cambiar tipo de entorno de ejecución"
!pip -q install ultralytics==8.4.218 roboflow
import os, sys, json, glob, time, shutil, subprocess, yaml, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from collections import Counter
from ultralytics import YOLO
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO (active la T4)')

## 2. Configuración (editable)
`MAPEO_M2` define el esquema de 4 categorías del peaje (nombres en español, los mismos que normaliza el
servidor `inferencia/peaje_ia/m2_vehiculos.py`). Las clases en `DESCARTAR_M2` se eliminan de las etiquetas.
Cualquier clase del dataset que no esté en ninguna de las dos listas también se descarta, con aviso.

In [ ]:
RF_WORKSPACE, RF_PROYECTO, RF_VERSION, RF_FORMATO = 'akash-bauri-nyofc', 'vehicle-segmentation', 4, 'yolov11'
DATASET_INFO = {'titulo': 'Vehicle Segmentation', 'autor': 'Akash Bauri', 'licencia': 'CC BY 4.0',
                'url': 'https://universe.roboflow.com/akash-bauri-nyofc/vehicle-segmentation',
                'workspace': RF_WORKSPACE, 'proyecto': RF_PROYECTO, 'version': RF_VERSION, 'formato': RF_FORMATO}

MAPEO_M2 = {
    'auto':    ['car', 'sedan', 'hatchback', 'suv', 'swift', 'thar', 'scorpio', 'bolero', 'pickup'],
    'minibus': ['van', 'omni'],
    'bus':     ['bus'],
    'camion':  ['truck'],
}
# 'auto' del dataset = auto-rickshaw (tuk-tuk indio): NO es un automóvil -> se descarta
DESCARTAR_M2 = ['auto', 'motorbike', 'bike-rider', 'motorbike-rider', 'scooty', 'scooty-rider', 'apache',
                'bullet', 'pulsar', 'person', 'man', 'woman', 'child', 'helmet', 'no-helmet', 'license-plate',
                'tractor', 'jcb']
ORDEN_CATEGORIAS = ['auto', 'minibus', 'bus', 'camion']
MIN_INSTANCIAS_TRAIN, MIN_CATEGORIAS = 30, 3

EPOCHS, IMGSZ, BATCH, PATIENCE, TOPE_HORAS = 40, 640, 16, 10, 0.9   # TOPE_HORAS: seguridad para Colab T4
REPO_URL = 'https://github.com/jayutomas623/sistema-peaje-ia'
DIR_DRIVE = '/content/drive/MyDrive/peaje-ia'
DIR_FINAL = '/content/m2_dataset'
M2_PT = '/content/m2_vehiculos_seg.pt'

In [ ]:
# === UTILIDADES COLAB (API key, Drive, servidor de inferencia del repo) ===
def obtener_api_key():
    # 1) Secreto de Colab 'ROBOFLOW_API_KEY' (🔑 en la barra lateral, con acceso habilitado para este notebook)
    # 2) Si no existe, se pide con getpass (no queda escrita en el notebook ni en el repositorio)
    try:
        from google.colab import userdata
        clave = userdata.get('ROBOFLOW_API_KEY')
        if clave:
            return clave
    except Exception as e:
        print('Secreto ROBOFLOW_API_KEY no disponible (' + type(e).__name__ + '): se pedirá a mano.')
    from getpass import getpass
    return getpass('Pegue su ROBOFLOW_API_KEY (no se guarda): ').strip()

def numero_version(v):
    # Objeto Version del SDK de Roboflow -> número entero (tolera distintas versiones del SDK)
    for valor in (getattr(v, 'version', None), getattr(v, 'id', None), v):
        try:
            return int(str(valor).rstrip('/').split('/')[-1])
        except (TypeError, ValueError):
            continue
    return -1

def descargar_roboflow(workspace, proyecto, version, formato):
    from roboflow import Roboflow
    rf = Roboflow(api_key=obtener_api_key())
    p = rf.workspace(workspace).project(proyecto)
    try:
        versiones = sorted(numero_version(v) for v in p.versions())
    except Exception as e:
        versiones = []; print('No se pudo listar versiones:', e)
    print(f'{workspace}/{proyecto} · versiones disponibles: {versiones} · se usa v{version}')
    if versiones and max(versiones) != version:
        print(f'⚠️  La última versión publicada es v{max(versiones)}. Se mantiene v{version} (la documentada en '
              f'entrenamiento/DATASETS.md); si cambia RF_VERSION, actualice también DATASETS.md.')
    return p.version(version).download(formato).location

def clonar_repo(destino='/content/sistema-peaje-ia'):
    if not os.path.isdir(destino):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, destino], check=True)
    return destino

def importar_servidor(repo_dir, **env):
    # Importa inferencia/peaje_ia (el MISMO código que corre en RunPod) con las variables de entorno dadas
    for k, v in env.items():
        os.environ[k] = str(v)
    for m in [m for m in sys.modules if m == 'peaje_ia' or m.startswith('peaje_ia.')]:
        del sys.modules[m]                      # recarga config.py con las nuevas rutas de pesos
    ruta = os.path.join(repo_dir, 'inferencia')
    if ruta not in sys.path:
        sys.path.insert(0, ruta)

def guardar_en_drive(archivos, subcarpeta=''):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive no montado:', e); return
    for a in archivos:
        if not os.path.exists(a):
            print('  (no existe, se omite)', a); continue
        # pesos -> MyDrive/peaje-ia/pesos/ · métricas, CSV y gráficos -> MyDrive/peaje-ia/<subcarpeta>/
        dest = os.path.join(DIR_DRIVE, 'pesos' if a.endswith('.pt') else subcarpeta)
        os.makedirs(dest, exist_ok=True)
        shutil.copy(a, dest); print('  ->', os.path.join(dest, os.path.basename(a)))

def tope_de_tiempo(horas):
    # Tope de seguridad para Colab: detiene el entrenamiento al terminar la época en que se supera `horas`.
    # (No se usa el argumento `time=` de Ultralytics porque recalcula `epochs` para llenar el tiempo.)
    inicio = time.time()
    def cb(trainer):
        if time.time() - inicio > horas * 3600:
            print(f'⏱️  Tope de {horas} h alcanzado: se detiene tras la época {trainer.epoch + 1}.')
            trainer.stop = True
    return cb

In [ ]:
# === FUNCIONES M2: remapeo de clases (probadas localmente con datos sintéticos) ===
import os, glob, shutil
from collections import Counter

def normalizar_nombre(n):
    return str(n).strip().lower().replace('_', '-').replace(' ', '-')

def nombres_de_yaml(cfg):
    n = cfg['names']
    return list(n) if isinstance(n, (list, tuple)) else [n[k] for k in sorted(n, key=int)]

def tabla_origen_destino(nombres, mapeo, descartar):
    # id de clase de origen -> categoría destino (str) o None (descartada). También devuelve las no previstas.
    fuente = {normalizar_nombre(o): destino for destino, origenes in mapeo.items() for o in origenes}
    desc = {normalizar_nombre(d) for d in descartar}
    choque = desc & set(fuente)
    assert not choque, f'Clases a la vez mapeadas y descartadas: {sorted(choque)}'
    tabla, no_previstas = {}, []
    for i, n in enumerate(nombres):
        k = normalizar_nombre(n)
        if k in fuente:
            tabla[i] = fuente[k]
        else:
            tabla[i] = None
            if k not in desc:
                no_previstas.append(n)
    return tabla, no_previstas

def es_poligono(tokens):
    # clase + al menos 3 puntos (x y) = 7 tokens, y número par de coordenadas
    return len(tokens) >= 7 and (len(tokens) - 1) % 2 == 0

def remapear_linea(linea, tabla, id_final):
    # -> (línea nueva | None, motivo). Solo cambia el id de clase: los polígonos se conservan tal cual.
    t = linea.split()
    if not t:
        return None, 'vacia'
    try:
        c = int(float(t[0]))
    except ValueError:
        return None, 'id_invalido'
    if c not in tabla:
        return None, 'id_invalido'
    if tabla[c] is None:
        return None, 'descartada'
    if not es_poligono(t):
        return None, 'no_poligono'
    if tabla[c] not in id_final:
        return None, 'categoria_excluida'
    return ' '.join([str(id_final[tabla[c]])] + t[1:]), 'ok'

def leer_etiquetas(raiz, split):
    for f in sorted(glob.glob(os.path.join(raiz, split, 'labels', '*.txt'))):
        with open(f, encoding='utf-8') as fh:
            yield f, fh.read().splitlines()

def contar_origen(raiz, splits, nombres):
    # instancias por split y clase de ORIGEN (todas las líneas)
    res = {s: Counter() for s in splits}
    for s in splits:
        for _, lineas in leer_etiquetas(raiz, s):
            for l in lineas:
                t = l.split()
                if t:
                    i = int(float(t[0]))
                    res[s][nombres[i] if 0 <= i < len(nombres) else f'id{i}'] += 1
    return res

def contar_destino(raiz, splits, tabla):
    # instancias por split y categoría DESTINO (solo polígonos válidos de clases mapeadas)
    res = {s: Counter() for s in splits}
    for s in splits:
        for _, lineas in leer_etiquetas(raiz, s):
            for l in lineas:
                t = l.split()
                if t and tabla.get(int(float(t[0]))) and es_poligono(t):
                    res[s][tabla[int(float(t[0]))]] += 1
    return res

def elegir_categorias(conteo_train, orden, minimo, min_categorias):
    usadas = [c for c in orden if conteo_train.get(c, 0) >= minimo]
    avisos = [f'"{c}" solo tiene {conteo_train.get(c, 0)} instancias en train (< {minimo}): '
              f'se entrena SIN esta categoría.' for c in orden if c not in usadas]
    if len(usadas) < min_categorias:
        raise RuntimeError(f'Solo {len(usadas)} categorías con >= {minimo} instancias en train ({usadas}); '
                           f'se exigen al menos {min_categorias}. Revise MAPEO_M2 o cambie de dataset.')
    return usadas, avisos

def _enlazar_o_copiar(origen, destino):
    try:
        os.symlink(os.path.abspath(origen), destino, target_is_directory=True)
    except (OSError, NotImplementedError):
        shutil.copytree(origen, destino)

def escribir_dataset(origen, destino, splits, tabla, usadas):
    # Dataset final en `destino`: imágenes enlazadas (sin copiar) y etiquetas reescritas.
    # Las imágenes que quedan sin objetos se conservan con un .txt vacío (fondo / negativos).
    id_final = {c: i for i, c in enumerate(usadas)}
    motivos = {s: Counter() for s in splits}
    resumen = {}
    shutil.rmtree(destino, ignore_errors=True)
    for s in splits:
        dir_img = os.path.join(origen, s, 'images')
        os.makedirs(os.path.join(destino, s, 'labels'))
        _enlazar_o_copiar(dir_img, os.path.join(destino, s, 'images'))
        n_img = n_fondo = 0
        for nombre in sorted(os.listdir(dir_img)):
            base = os.path.splitext(nombre)[0]
            lbl = os.path.join(origen, s, 'labels', base + '.txt')
            lineas = []
            if os.path.exists(lbl):
                with open(lbl, encoding='utf-8') as fh:
                    lineas = fh.read().splitlines()
            nuevas = []
            for l in lineas:
                nl, m = remapear_linea(l, tabla, id_final)
                if m != 'vacia':
                    motivos[s][m] += 1
                if nl:
                    nuevas.append(nl)
            with open(os.path.join(destino, s, 'labels', base + '.txt'), 'w', encoding='utf-8') as fh:
                fh.write('\n'.join(nuevas) + ('\n' if nuevas else ''))
            n_img += 1
            n_fondo += not nuevas
        resumen[s] = {'imagenes': n_img, 'imagenes_fondo': n_fondo}
    return id_final, motivos, resumen

def yaml_final(destino, splits, usadas):
    d = {'path': os.path.abspath(destino), 'train': 'train/images', 'val': 'valid/images',
         'nc': len(usadas), 'names': {i: c for i, c in enumerate(usadas)}}
    if 'test' in splits:
        d['test'] = 'test/images'
    return d

## 3. Descarga del dataset y conteo de instancias por clase y split

In [ ]:
DS_DIR = descargar_roboflow(RF_WORKSPACE, RF_PROYECTO, RF_VERSION, RF_FORMATO)
SPLITS = [s for s in ('train', 'valid', 'test') if os.path.isdir(f'{DS_DIR}/{s}/labels')]
print(open(f'{DS_DIR}/data.yaml').read())
cfg_origen = yaml.safe_load(open(f'{DS_DIR}/data.yaml'))
NOMBRES_ORIGEN = nombres_de_yaml(cfg_origen)
print('names:', NOMBRES_ORIGEN)
print('splits:', SPLITS)

conteo_origen = contar_origen(DS_DIR, SPLITS, NOMBRES_ORIGEN)
df_origen = pd.DataFrame(conteo_origen).fillna(0).astype(int)
df_origen['total'] = df_origen.sum(axis=1)
df_origen.sort_values('total', ascending=False)

### 3.1 Verificación visual: la clase `auto` del dataset son rickshaws
Se dibujan las máscaras (polígonos de las etiquetas) de la clase `auto` en 3 imágenes de train.

In [ ]:
def mostrar_clase(raiz, split, nombres, clase, n=3):
    ids = [i for i, x in enumerate(nombres) if normalizar_nombre(x) == normalizar_nombre(clase)]
    if not ids:
        print(f'La clase "{clase}" no existe en este dataset.'); return
    elegidas = []
    for f, lineas in leer_etiquetas(raiz, split):
        polis = [l.split() for l in lineas if l.split() and int(float(l.split()[0])) in ids]
        if polis:
            elegidas.append((f, polis))
        if len(elegidas) == n:
            break
    fig, ax = plt.subplots(1, len(elegidas), figsize=(6 * len(elegidas), 5))
    for a, (f, polis) in zip(np.atleast_1d(ax), elegidas):
        base = os.path.splitext(os.path.basename(f))[0]
        ruta = next(iter(glob.glob(f'{raiz}/{split}/images/{glob.escape(base)}.*')))
        img = cv2.imread(ruta); H, W = img.shape[:2]; capa = img.copy()
        for t in polis:
            if not es_poligono(t): continue
            pts = (np.array(t[1:], float).reshape(-1, 2) * [W, H]).astype(np.int32)
            cv2.fillPoly(capa, [pts], (0, 140, 255)); cv2.polylines(img, [pts], True, (0, 140, 255), 3)
        a.imshow(cv2.addWeighted(capa, 0.45, img, 0.55, 0)[:, :, ::-1]); a.axis('off')
        a.set_title(f'clase "{clase}" · {len(polis)} instancia(s)')
    plt.suptitle(f'Clase "{clase}" del dataset (¿auto-rickshaw?)'); plt.show()

mostrar_clase(DS_DIR, 'train', NOMBRES_ORIGEN, 'auto')

## 4. Remapeo al esquema del peaje y selección de categorías

In [ ]:
TABLA, NO_PREVISTAS = tabla_origen_destino(NOMBRES_ORIGEN, MAPEO_M2, DESCARTAR_M2)
if NO_PREVISTAS:
    print('⚠️  Clases del dataset que no están en MAPEO_M2 ni en DESCARTAR_M2 (se DESCARTAN):', NO_PREVISTAS)
print(pd.DataFrame([{'id': i, 'clase_origen': n, 'categoria_peaje': TABLA[i] or '— descartada —'}
                    for i, n in enumerate(NOMBRES_ORIGEN)]).to_string(index=False))

conteo_destino = contar_destino(DS_DIR, SPLITS, TABLA)
CATEGORIAS, AVISOS = elegir_categorias(conteo_destino['train'], ORDEN_CATEGORIAS, MIN_INSTANCIAS_TRAIN, MIN_CATEGORIAS)
for a in AVISOS:
    print('⚠️  AVISO:', a)
print('Categorías que se entrenan:', CATEGORIAS)

ID_FINAL, MOTIVOS, RESUMEN_IMGS = escribir_dataset(DS_DIR, DIR_FINAL, SPLITS, TABLA, CATEGORIAS)
DATA_FINAL = f'{DIR_FINAL}/data.yaml'
yaml.safe_dump(yaml_final(DIR_FINAL, SPLITS, CATEGORIAS), open(DATA_FINAL, 'w'), allow_unicode=True, sort_keys=False)
print(open(DATA_FINAL).read())

df_motivos = pd.DataFrame(MOTIVOS).fillna(0).astype(int)
print('Líneas de etiqueta por motivo (ok = conservadas):'); display(df_motivos)
print('Imágenes por split (las de fondo se conservan sin objetos):'); display(pd.DataFrame(RESUMEN_IMGS))
INSTANCIAS_FINALES = {s: dict(c) for s, c in contar_origen(DIR_FINAL, SPLITS, CATEGORIAS).items()}
display(pd.DataFrame(INSTANCIAS_FINALES).fillna(0).astype(int))

## 5. Entrenamiento YOLO11s-seg
`epochs=40`, `patience=10`, `imgsz=640`, `batch=16`. El tope de **0.9 h** se aplica con un *callback* que
detiene el entrenamiento al terminar la época en que se supera (el argumento `time=` de Ultralytics no se usa
porque recalcula `epochs` para llenar todo el tiempo disponible).

In [ ]:
modelo = YOLO('yolo11s-seg.pt')
modelo.add_callback('on_fit_epoch_end', tope_de_tiempo(TOPE_HORAS))
t0 = time.time()
modelo.train(data=DATA_FINAL, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE, cos_lr=True,
             project='/content/runs/segment', name='m2', exist_ok=True, device=DEVICE, plots=True)
TIEMPO_MIN = round((time.time() - t0) / 60, 1)
RUN_DIR = str(modelo.trainer.save_dir)
EPOCAS_REALIZADAS = len(pd.read_csv(f'{RUN_DIR}/results.csv'))
print(f'Entrenamiento: {TIEMPO_MIN} min · {EPOCAS_REALIZADAS} épocas · {RUN_DIR}')

## 6. Métricas: mAP50 y mAP50-95 de caja y máscara (global y por clase) + matriz de confusión

In [ ]:
best = YOLO(f'{RUN_DIR}/weights/best.pt')

def metricas(split):
    v = best.val(data=DATA_FINAL, split=split, imgsz=IMGSZ, plots=True, device=DEVICE,
                 project='/content/runs/segment', name=f'm2_{split}', exist_ok=True)
    g = {'mAP50_box': float(v.box.map50), 'mAP50_95_box': float(v.box.map),
         'mAP50_mask': float(v.seg.map50), 'mAP50_95_mask': float(v.seg.map),
         'precision_mask': float(v.seg.mp), 'recall_mask': float(v.seg.mr)}
    pc = [{'clase': best.names[int(c)], 'mAP50_box': float(v.box.ap50[j]), 'mAP50_95_box': float(v.box.ap[j]),
           'mAP50_mask': float(v.seg.ap50[j]), 'mAP50_95_mask': float(v.seg.ap[j])}
          for j, c in enumerate(v.box.ap_class_index)]
    return v, g, pc

val_v, MET_VALID, POR_CLASE_VALID = metricas('valid')
display(pd.DataFrame([MET_VALID]).round(4)); display(pd.DataFrame(POR_CLASE_VALID).round(4))
MET_TEST = POR_CLASE_TEST = None
if 'test' in SPLITS:
    _, MET_TEST, POR_CLASE_TEST = metricas('test')
    print('Split test:'); display(pd.DataFrame([MET_TEST]).round(4)); display(pd.DataFrame(POR_CLASE_TEST).round(4))

try:
    MATRIZ = {'nombres': [best.names[i] for i in sorted(best.names)] + ['background'],
              'matriz': val_v.confusion_matrix.matrix.astype(int).tolist()}
except AttributeError as e:
    MATRIZ = None; print('Matriz de confusión numérica no disponible en esta versión:', e)
from IPython.display import Image
for f in ['confusion_matrix_normalized.png', 'confusion_matrix.png', 'MaskPR_curve.png']:
    p = os.path.join(str(val_v.save_dir), f)
    if os.path.exists(p): display(Image(p, width=650))
display(Image(f'{RUN_DIR}/results.png', width=1000))

## 7. Exportar `m2_vehiculos_seg.pt`

In [ ]:
shutil.copy(f'{RUN_DIR}/weights/best.pt', M2_PT)
print('Clases del modelo:', YOLO(M2_PT).names)

## 8. Evaluación extra en las 22 fotos bolivianas del repositorio
Se clona el repositorio y se usa **la misma clase del servidor** (`SegmentadorVehiculos` de
`inferencia/peaje_ia/m2_vehiculos.py`) con `M2_PESOS` apuntando al modelo recién entrenado: así se comprueba
que el `.pt` carga en RunPod sin cambios y se elige el vehículo principal con la misma regla
(mayor área de máscara × confianza). La categoría real viene de `web/database/seeders/evidencias/manifest.csv`.

> Las fotos bolivianas son casi todas **frontales** y el dataset es de la India: este resultado mide la
> transferencia a nuestro dominio y se reporta tal cual, sea bueno o malo.

In [ ]:
REPO_DIR = clonar_repo()
EVID = f'{REPO_DIR}/web/database/seeders/evidencias'
manifest = pd.read_csv(f'{EVID}/manifest.csv')
print('Columnas de manifest.csv:', manifest.columns.tolist())
COL_ARCHIVO = next(c for c in ('archivo', 'imagen', 'file', 'filename') if c in manifest.columns)
COL_CATEGORIA = next(c for c in ('categoria', 'categoría', 'category', 'clase') if c in manifest.columns)

importar_servidor(REPO_DIR, M2_PESOS=M2_PT, DEVICE='auto')
from peaje_ia.m2_vehiculos import SegmentadorVehiculos
seg = SegmentadorVehiculos()
assert seg.modo == 'entrenado', f'El servidor no cargó el modelo entrenado: {seg.nombre}'
print('Servidor:', seg.nombre)

filas = []
for _, fila in manifest.iterrows():
    img = cv2.imread(os.path.join(EVID, fila[COL_ARCHIVO]))
    principal, _ = seg.inferir(img)
    pred = principal['categoria'] if principal else 'sin_deteccion'
    filas.append({'archivo': fila[COL_ARCHIVO], 'categoria_real': str(fila[COL_CATEGORIA]).strip().lower(),
                  'categoria_predicha': pred,
                  'confianza': round(principal['conf'], 3) if principal else None,
                  'area_mascara': round(principal['area_mascara'], 3) if principal else None})
df_bo = pd.DataFrame(filas)
df_bo['acierto'] = df_bo.categoria_real == df_bo.categoria_predicha
df_bo.to_csv('/content/m2_eval_bolivia.csv', index=False)
EXACTITUD_BO = float(df_bo.acierto.mean())
POR_CAT_BO = (df_bo.groupby('categoria_real').acierto.agg(['mean', 'count'])
              .rename(columns={'mean': 'exactitud', 'count': 'n'}).reset_index().to_dict('records'))
fuera = sorted(set(df_bo.categoria_real) - set(CATEGORIAS))
if fuera:
    print('⚠️  Categorías reales que este modelo NO puede predecir (no se entrenaron):', fuera)
print(f'Exactitud global en fotos bolivianas: {EXACTITUD_BO:.1%} (n = {len(df_bo)})')
display(pd.DataFrame(POR_CAT_BO)); display(pd.crosstab(df_bo.categoria_real, df_bo.categoria_predicha))
df_bo

In [ ]:
muestras = df_bo.sample(min(6, len(df_bo)), random_state=0)
fig, ax = plt.subplots(2, 3, figsize=(18, 10)); ax = ax.ravel()
for a, (_, f) in zip(ax, muestras.iterrows()):
    r = best.predict(os.path.join(EVID, f.archivo), conf=0.35, device=DEVICE, verbose=False)[0]
    a.imshow(r.plot()[:, :, ::-1]); a.axis('off')
    a.set_title(f'real: {f.categoria_real} · pred: {f.categoria_predicha}', color='green' if f.acierto else 'red')
plt.tight_layout(); plt.show()

## 9. `m2_metricas.json` y copia en Google Drive

In [ ]:
metricas_json = {
    'modelo': 'yolo11s-seg fine-tuning', 'archivo': 'm2_vehiculos_seg.pt',
    'dataset': DATASET_INFO,
    'mapeo': {'categorias_peaje': MAPEO_M2, 'descartadas': DESCARTAR_M2, 'no_previstas_descartadas': NO_PREVISTAS,
              'tabla_id_origen': {str(i): {'origen': n, 'destino': TABLA[i]} for i, n in enumerate(NOMBRES_ORIGEN)}},
    'categorias_entrenadas': CATEGORIAS, 'avisos': AVISOS,
    'instancias': {'origen_por_split': {s: dict(c) for s, c in conteo_origen.items()},
                   'final_por_split': INSTANCIAS_FINALES},
    'lineas_por_motivo': {s: dict(c) for s, c in MOTIVOS.items()}, 'imagenes': RESUMEN_IMGS,
    'entrenamiento': {'epochs': EPOCHS, 'epocas_realizadas': EPOCAS_REALIZADAS, 'patience': PATIENCE,
                      'imgsz': IMGSZ, 'batch': BATCH, 'tope_horas': TOPE_HORAS, 'tiempo_min': TIEMPO_MIN,
                      'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'},
    'metricas_valid': {'global': MET_VALID, 'por_clase': POR_CLASE_VALID},
    'metricas_test': {'global': MET_TEST, 'por_clase': POR_CLASE_TEST} if MET_TEST else None,
    'matriz_confusion_valid': MATRIZ,
    'bolivia': {'n': len(df_bo), 'exactitud_global': EXACTITUD_BO, 'por_categoria': POR_CAT_BO,
                'csv': 'm2_eval_bolivia.csv', 'nota': 'Vehículo principal elegido con SegmentadorVehiculos del servidor'},
}
json.dump(metricas_json, open('/content/m2_metricas.json', 'w'), indent=2, ensure_ascii=False)
_png = os.path.join(str(val_v.save_dir), 'confusion_matrix_normalized.png')
if os.path.exists(_png):
    shutil.copy(_png, '/content/m2_matriz_confusion.png')
guardar_en_drive([M2_PT, '/content/m2_metricas.json', '/content/m2_eval_bolivia.csv',
                  '/content/m2_matriz_confusion.png', f'{RUN_DIR}/results.png'], subcarpeta='metricas/m2')
print(json.dumps({k: metricas_json[k] for k in ('categorias_entrenadas', 'avisos', 'metricas_valid')}, indent=2)[:3000])